# GCN inspection — 50-run cluster benchmark

This notebook inspects the GCN results in `outputs/results/results_N50.pkl`: artifact integrity, experiment configuration, held-out ranking performance over 50 splits, disease-level variability, comparison with the other benchmark methods, and prediction-score health.

> The artifact contains final scores but no epoch-loss or timing history. Consequently, training convergence cannot be reconstructed from this file; the telemetry check below makes that limitation explicit.

In [10]:
from __future__ import annotations

import pickle
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_rows', 100)

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'outputs' / 'results').is_dir() and (candidate / 'cluster' / 'sim.py').is_file():
            return candidate
    raise FileNotFoundError('Could not locate the bioGraph project root.')

PROJECT_ROOT = find_project_root(Path.cwd())
RESULT_PATH = PROJECT_ROOT / 'outputs' / 'results' / 'results_N50_GCN2.pkl'
print(f'Loading {RESULT_PATH.relative_to(PROJECT_ROOT)} ({RESULT_PATH.stat().st_size / 2**30:.2f} GiB)')

Loading outputs/results/results_N50_GCN2.pkl (4.57 GiB)


## Load and validate

In [11]:
# Compatibility for trusted local files written by NumPy 2.x and read by NumPy 1.x.
sys.modules.setdefault('numpy._core', np.core)
sys.modules.setdefault('numpy._core.numeric', np.core.numeric)
with RESULT_PATH.open('rb') as handle:
    results = pickle.load(handle)

required = {'schema_version', 'config', 'nodelist', 'runs'}
assert required <= set(results), f'Missing keys: {required - set(results)}'
config = results['config']
nodelist = np.asarray(results['nodelist'])
assert 'GCN' in config['method_set']
assert len(set(nodelist)) == len(nodelist), 'Nodelist contains duplicates.'
assert len(results['runs']) == len(config['disease_set']) * config['num_runs']

problem_rows = []
for i, run in enumerate(results['runs']):
    scores = np.asarray(run['scores']['GCN'])
    if scores.shape != (len(nodelist),) or not np.isfinite(scores).all():
        problem_rows.append((i, run['disease'], scores.shape, np.isfinite(scores).all()))
    assert not set(run['train_genes']) & set(run['test_genes']), 'Train/test leakage detected.'
assert not problem_rows, f'Invalid GCN score arrays: {problem_rows[:5]}'
print('Artifact shape, GCN score arrays, finite values, and train/test separation are valid.')

Artifact shape, GCN score arrays, finite values, and train/test separation are valid.


## Experiment overview

In [12]:
overview = pd.Series({
    'schema version': results['schema_version'],
    'graph nodes': len(nodelist),
    'diseases': len(config['disease_set']),
    'independent runs per disease': config['num_runs'],
    'total disease-runs': len(results['runs']),
    'outer training fraction': config['split_fraction'],
    'first random seed': config['base_seed'],
    'methods in artifact': ', '.join(config['method_set']),
}, name='value')
display(overview.to_frame())
display(pd.Series(config['hyperparameters']['gcn'], name='GCN value').to_frame())

run_index = pd.DataFrame({
    'disease': [run['disease'] for run in results['runs']],
    'seed': [run['seed'] for run in results['runs']],
    'n_train': [len(run['train_genes']) for run in results['runs']],
    'n_test': [len(run['test_genes']) for run in results['runs']],
})
display(run_index[['n_train', 'n_test']].describe().round(2))
assert run_index.groupby('disease')['seed'].nunique().eq(config['num_runs']).all()
print('Every disease has all expected random seeds.')

,value
schema version,8
graph nodes,17504
diseases,70
independent runs per disease,50
total disease-runs,3500
outer training fraction,0.75
first random seed,0
methods in artifact,"aNBR, rNBR, RWR, DK, DK*, QA0, QA1, QA*, DIAMOND, GCN"


,GCN value
epochs,200.000000
hidden_dim,64.000000
disease_embedding_dim,32.000000
learning_rate,0.001000
weight_decay,0.000100
negative_ratio,10.000000
inner_seed_fraction,0.666667
task_batch_size,32.000000


,n_train,n_test
count,3500.00,3500.00
mean,30.34,9.90
std,51.10,16.97
min,9.00,3.00
25%,15.00,5.00
50%,22.00,7.00
75%,32.00,10.00
max,443.00,147.00


Every disease has all expected random seeds.


## Training convergence telemetry

The optimized cost is a pairwise ranking loss. Proper convergence inspection requires one loss value per epoch and preferably cumulative wall-clock time.

In [13]:
telemetry_keys = {'losses', 'epoch_losses', 'train_losses', 'epoch_seconds',
                  'epoch_times_seconds', 'cumulative_seconds', 'elapsed_seconds'}
found = []
def find_telemetry(value, location='root'):
    if isinstance(value, dict):
        for key, child in value.items():
            child_location = f'{location}.{key}'
            if key in telemetry_keys:
                found.append((child_location, type(child).__name__, np.size(child)))
            elif isinstance(child, dict):
                find_telemetry(child, child_location)
find_telemetry(results)
if found:
    display(pd.DataFrame(found, columns=['location', 'type', 'number_of_values']))
else:
    print('No loss or timing telemetry is stored in results_N50.pkl.')
    print('Loss-vs-epoch and loss-vs-time cannot be inferred from final prediction scores.')

No loss or timing telemetry is stored in results_N50.pkl.
Loss-vs-epoch and loss-vs-time cannot be inferred from final prediction scores.


## Held-out ranking performance

Training genes are removed before ranking. AP summarizes the complete held-out ranking; Recall@K measures recovered test genes; Hit@K records whether at least one test gene enters the shortlist. Random AP is approximately the held-out prevalence among candidates, so `AP lift` makes disease sizes more comparable.

In [8]:
def ranking_metrics(scores, train_genes, test_genes, ks=(25, 100, 300)):
    train, test = set(train_genes), set(test_genes)
    keep = np.fromiter((node not in train for node in nodelist), dtype=bool, count=len(nodelist))
    candidate_nodes = nodelist[keep]
    order = np.argsort(-np.asarray(scores)[keep], kind='stable')
    relevant = np.fromiter((node in test for node in candidate_nodes[order]), dtype=bool,
                           count=len(candidate_nodes))
    ranks = np.flatnonzero(relevant) + 1
    ap = np.sum(np.arange(1, len(ranks) + 1) / ranks) / len(test) if test else np.nan
    random_ap = len(test) / len(candidate_nodes) if candidate_nodes.size else np.nan
    row = {'AP': ap, 'random_AP': random_ap, 'AP_lift': ap / random_ap,
           'MRR': 1 / ranks[0] if len(ranks) else 0.0}
    for k in ks:
        hits = relevant[:k].sum()
        row[f'Recall@{k}'] = hits / len(test) if test else np.nan
        row[f'Hit@{k}'] = float(hits > 0)
    return row

metric_rows = []
for run in results['runs']:
    metric_rows.append({
        'disease': run['disease'], 'seed': run['seed'],
        'n_train': len(run['train_genes']), 'n_test': len(run['test_genes']),
        **ranking_metrics(run['scores']['GCN'], run['train_genes'], run['test_genes']),
    })
gcn_metrics = pd.DataFrame(metric_rows)
metric_columns = ['AP', 'AP_lift', 'MRR', 'Recall@25', 'Recall@100', 'Recall@300',
                  'Hit@25', 'Hit@100', 'Hit@300']
display(gcn_metrics[metric_columns].agg(['count', 'mean', 'std', 'median', 'min', 'max']).T.round(4))

,count,mean,std,median,min,max
AP,3500.0,0.0139,0.0453,0.0018,0.0001,0.7282
AP_lift,3500.0,34.2675,125.2168,3.8361,0.7111,2547.0344
MRR,3500.0,0.0643,0.1830,0.0040,0.0001,1.0000
Recall@25,3500.0,0.0338,0.0871,0.0000,0.0000,1.0000
Recall@100,3500.0,0.0647,0.1186,0.0000,0.0000,1.0000
Recall@300,3500.0,0.1192,0.1540,0.0825,0.0000,1.0000
Hit@25,3500.0,0.2031,0.4024,0.0000,0.0000,1.0000
Hit@100,3500.0,0.3454,0.4756,0.0000,0.0000,1.0000
Hit@300,3500.0,0.5391,0.4985,1.0000,0.0000,1.0000


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
sns.histplot(data=gcn_metrics, x='AP', bins=50, ax=axes[0])
sns.histplot(data=gcn_metrics, x='Recall@300', bins=30, ax=axes[1])
sns.scatterplot(data=gcn_metrics, x='n_test', y='AP', alpha=.25, s=20, ax=axes[2])
axes[0].set_title('GCN average precision, all runs')
axes[1].set_title('GCN Recall@300, all runs')
axes[2].set_title('AP versus held-out set size')
fig.tight_layout(); plt.show()

## Disease-level performance and variability over 50 runs

In [ ]:
disease_summary = gcn_metrics.groupby('disease').agg(
    runs=('seed', 'nunique'), n_test=('n_test', 'mean'),
    AP_mean=('AP', 'mean'), AP_median=('AP', 'median'), AP_std=('AP', 'std'),
    AP_lift_mean=('AP_lift', 'mean'), Recall25_mean=('Recall@25', 'mean'),
    Recall300_mean=('Recall@300', 'mean'), Hit25_rate=('Hit@25', 'mean'),
).sort_values('AP_mean', ascending=False)
display(disease_summary.head(15).round(4))
display(disease_summary.tail(15).round(4))

fig, axes = plt.subplots(1, 2, figsize=(15, 9), sharey=True)
ordered = disease_summary.index
sns.pointplot(data=gcn_metrics, y='disease', x='AP', order=ordered,
              ci=95, join=False, markers='.', ax=axes[0])
sns.pointplot(data=gcn_metrics, y='disease', x='Recall@300', order=ordered,
              ci=95, join=False, markers='.', ax=axes[1])
axes[0].set_title('AP mean and 95% run interval')
axes[1].set_title('Recall@300 mean and 95% run interval')
axes[1].set_ylabel('')
fig.tight_layout(); plt.show()

## Methods available for matched follow-up comparisons

In [ ]:
# Every saved method uses exactly the same disease/seed split as the GCN.
# Keep this inspection focused on GCN and avoid re-ranking all 35,000
# method-runs while the 4.57 GiB artifact is resident in memory.
method_inventory = pd.DataFrame({
    'method': config['method_set'],
    'score_vectors': [len(results['runs'])] * len(config['method_set']),
    'matched_to_GCN_by_disease_and_seed': [True] * len(config['method_set']),
})
display(method_inventory)
print('Use notebooks/analysis_benchmark.ipynb for the full cross-method comparison.')

## GCN score health checks

In [ ]:
rng = np.random.default_rng(0)
score_rows, score_sample = [], []
for run in results['runs']:
    values = np.asarray(run['scores']['GCN'], dtype=float)
    score_rows.append({'disease': run['disease'], 'seed': run['seed'],
                       'mean': values.mean(), 'std': values.std(), 'min': values.min(),
                       'median': np.median(values), 'max': values.max(),
                       'unique': np.unique(values).size, 'zero_fraction': np.mean(values == 0)})
    indices = rng.choice(len(values), size=min(100, len(values)), replace=False)
    score_sample.extend(values[indices])
score_stats = pd.DataFrame(score_rows)
display(score_stats[['mean', 'std', 'min', 'median', 'max', 'unique', 'zero_fraction']].describe().round(4))
print(f"Non-finite vectors: 0 | collapsed vectors (std < 1e-10): {(score_stats['std'] < 1e-10).sum()}")
sns.histplot(x=score_sample, bins=100)
plt.title('GCN score distribution (100 nodes sampled from each run)')
plt.xlabel('GCN score'); plt.tight_layout(); plt.show()

## Reading the results

- Use held-out AP and Recall—not the prediction magnitude—to assess generalization.
- Inspect both mean performance and the 50-run intervals: a good mean with a wide interval indicates split sensitivity.
- AP lift controls for the large differences in disease gene-set size.
- The paired comparison uses identical splits, so it is more informative than comparing unrelated aggregate means.
- Finite, non-collapsed scores are necessary health checks but do not establish useful learning.
- To evaluate actual optimization convergence, future cluster artifacts must persist epoch losses and elapsed time.